# Week 50, Day 2: Breadth-first search

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w050d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from collections import deque

adj = {"A": ["B", "C"], "B": ["A", "D"], "C": ["A", "D"], "D": ["B", "C", "E"], "E": ["D"]}

order, dist = [], {"A": 0}
queue = deque(["A"])
while queue:
    v = queue.popleft()
    order.append(v)
    for w in adj[v]:
        if w not in dist:
            dist[w] = dist[v] + 1
            queue.append(w)
print(order)
print(dist)

In [ ]:
from collections import deque

def bfs_order(adj, start):
    seen, order = {start}, []
    queue = deque([start])
    while queue:
        v = queue.popleft()
        order.append(v)
        for w in adj[v]:
            if w not in seen:
                seen.add(w)
                queue.append(w)
    return order

adj = {1: [2, 3], 2: [4], 3: [4], 4: [1], 5: [6], 6: []}
print(bfs_order(adj, 1), bfs_order(adj, 5))

In [ ]:
from collections import deque

def shortest_path(adj, start, goal):
    parent = {start: None}
    queue = deque([start])
    while queue:
        v = queue.popleft()
        if v == goal:
            break
        for w in adj[v]:
            if w not in parent:
                parent[w] = v
                queue.append(w)
    if goal not in parent:
        return None
    path = []
    while goal is not None:
        path.append(goal)
        goal = parent[goal]
    return path[::-1]

flights = {"LOS": ["ACC", "LHR"], "ACC": ["NBO"], "LHR": ["NBO", "JFK"], "NBO": ["JNB"], "JFK": [], "JNB": []}
print(shortest_path(flights, "LOS", "JNB"), shortest_path(flights, "JFK", "LOS"))

In [ ]:
from collections import deque

def jug_moves(state, a_size=3, b_size=5):
    a, b = state
    pour_ab = min(a, b_size - b)
    pour_ba = min(b, a_size - a)
    return {(a_size, b), (a, b_size), (0, b), (a, 0), (a - pour_ab, b + pour_ab), (a + pour_ba, b - pour_ba)} - {state}

def solve(target):
    parent = {(0, 0): None}
    queue = deque([(0, 0)])
    while queue:
        s = queue.popleft()
        if target in s:
            path = []
            while s:
                path.append(s)
                s = parent[s]
            return path[::-1]
        for t in sorted(jug_moves(s)):
            if t not in parent:
                parent[t] = s
                queue.append(t)

print(solve(4))

In [ ]:
from collections import deque

town = ["S..#",
        "..#.",
        "...S"]

def nearest(grid):
    rows, cols = len(grid), len(grid[0])
    dist = {(r, c): 0 for r in range(rows) for c in range(cols) if grid[r][c] == "S"}
    queue = deque(dist)
    while queue:
        r, c = queue.popleft()
        for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                queue.append((nr, nc))
    return ["".join("#" if grid[r][c] == "#" else str(dist.get((r, c), "?")) for c in range(cols)) for r in range(rows)]

print("\n".join(nearest(town)))

In [ ]:
from collections import deque

def two_colour(adj):
    colour = {}
    for s in adj:                          # every piece of the graph
        if s in colour:
            continue
        colour[s] = 0
        queue = deque([s])
        while queue:
            v = queue.popleft()
            for w in adj[v]:
                if w not in colour:
                    colour[w] = 1 - colour[v]
                    queue.append(w)
                elif colour[w] == colour[v]:
                    return None             # an edge inside one group
    return colour

square = {1: [2, 4], 2: [1, 3], 3: [2, 4], 4: [3, 1]}
triangle = {1: [2, 3], 2: [1, 3], 3: [1, 2]}
print(two_colour(square), two_colour(triangle))

> **Key idea.** BFS takes vertices from the front of a deque and adds unseen neighbours to the back, marking them seen when added: O(V + E), visiting only what's reachable. In an unweighted graph, BFS reaches vertices in order of distance, so `dist[w] = dist[v] + 1` gives shortest distances, and parent links rebuild shortest paths (the BFS tree). Grids and puzzle states are graphs BFS searches unchanged. Putting many sources in the queue at distance 0 finds each vertex's nearest source in one pass. Colouring with BFS tests whether a graph is bipartite (no odd cycle). For weighted graphs, BFS isn't enough: that's Dijkstra's algorithm, next week.

### Exercise: Distances

Write `distances(adj, start)`: a dict from every vertex reachable from `start` to its distance in edges, by BFS with a `deque`. Then `levels(adj, start)`: a list of lists, the vertices at distance 0, 1, 2, ..., each list sorted. Then `farthest(adj, start)`: the largest distance and the sorted list of vertices at it.

In [ ]:
from collections import deque

def distances(adj, start):
    ___

def levels(adj, start):
    ___

def farthest(adj, start):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_dist():\n    \"distances\"\n    adj = {\"A\": [\"B\", \"C\"], \"B\": [\"A\", \"D\"], \"C\": [\"A\", \"D\"], \"D\": [\"B\", \"C\", \"E\"], \"E\": [\"D\"], \"F\": []}\n    assert distances(adj, \"A\") == {\"A\": 0, \"B\": 1, \"C\": 1, \"D\": 2, \"E\": 3} and distances(adj, \"F\") == {\"F\": 0}\n    assert levels(adj, \"A\") == [[\"A\"], [\"B\", \"C\"], [\"D\"], [\"E\"]] and farthest(adj, \"A\") == (3, [\"E\"])\n    rng = random.Random(1)\n    for _ in range(50):\n        n = rng.randint(1, 10)\n        g = {v: [w for w in range(n) if w != v and rng.random() < 0.25] for v in range(n)}\n        d = distances(g, 0)\n        # check: every edge changes the distance by at most 1, and each vertex but the start has a closer neighbour\n        for v in d:\n            for w in g[v]:\n                assert w in d and d[w] <= d[v] + 1\n            if v != 0:\n                assert any(v in g[u] and d[u] == d[v] - 1 for u in d)")

### Exercise: Shortest paths

Write `shortest_path(adj, start, goal)`: a shortest path as a list of vertices, or `None` if the goal can't be reached (a path from a vertex to itself is `[start]`). Explore neighbours in the order the adjacency list gives them. Then `all_shortest_counts(adj, start)`: a dict from each reachable vertex to the **number** of different shortest paths reaching it from `start` (a vertex at distance d + 1 gets the sum of the counts of its neighbours at distance d).

In [ ]:
from collections import deque

def shortest_path(adj, start, goal):
    ___

def all_shortest_counts(adj, start):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, itertools\ndef test_paths():\n    \"paths\"\n    flights = {\"LOS\": [\"ACC\", \"LHR\"], \"ACC\": [\"NBO\"], \"LHR\": [\"NBO\", \"JFK\"], \"NBO\": [\"JNB\"], \"JFK\": [], \"JNB\": []}\n    assert shortest_path(flights, \"LOS\", \"JNB\") == [\"LOS\", \"ACC\", \"NBO\", \"JNB\"] and shortest_path(flights, \"JFK\", \"LOS\") is None\n    assert shortest_path(flights, \"ACC\", \"ACC\") == [\"ACC\"]\n    grid4 = {(r, c): [(r + 1, c), (r, c + 1)] if r < 2 and c < 2 else [(r + 1, c)] if r < 2 else [(r, c + 1)] if c < 2 else [] for r in range(3) for c in range(3)}\n    assert all_shortest_counts(grid4, (0, 0))[(2, 2)] == 6\n    rng = random.Random(2)\n    for _ in range(60):\n        n = rng.randint(2, 7)\n        g = {v: [w for w in range(n) if w != v and rng.random() < 0.35] for v in range(n)}\n        counts = all_shortest_counts(g, 0)\n        for t in range(n):\n            paths = []\n            for L in range(n):\n                for mid in itertools.permutations([x for x in range(n) if x not in (0, t)], L):\n                    p = (0,) + mid + ((t,) if t else ())\n                    if all(b in g[a] for a, b in zip(p, p[1:])):\n                        paths.append(p)\n                if paths:\n                    break\n            if t == 0:\n                assert counts[0] == 1\n            elif paths:\n                assert counts[t] == len(paths) and len(shortest_path(g, 0, t)) == len(paths[0])\n            else:\n                assert t not in counts and shortest_path(g, 0, t) is None")

### Exercise: Through the maze

A maze is a list of strings: `#` walls, `.` open, one `S` (start) and one `E` (exit). Write `maze_steps(maze)`: the fewest moves (up, down, left, right) from S to E, or -1. Then `maze_route(maze)`: the maze as a list of strings with a shortest route's cells (not S or E) marked `*`, or `None` if there's no route; when several routes are shortest, explore neighbours in the order up, down, left, right.

In [ ]:
from collections import deque

def maze_steps(maze):
    ___

def maze_route(maze):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("MAZE = [\"S.#.....\",\n        \".##.###.\",\n        \"......#E\",\n        \".####.#.\",\n        \"........\"]\ndef test_maze():\n    \"mazes\"\n    assert maze_steps(MAZE) == 13\n    route = maze_route(MAZE)\n    assert route[0] == \"S.#.....\", route\n    assert sum(row.count(\"*\") for row in route) == 12 and all(len(a) == len(b) for a, b in zip(route, MAZE))\n    assert maze_steps([\"S#E\"]) == -1 and maze_route([\"S#E\"]) is None and maze_steps([\"SE\"]) == 1 and maze_route([\"SE\"]) == [\"SE\"]\n    big = [\"S\" + \".\" * 199] + [\".\" * 200 for _ in range(198)] + [\".\" * 199 + \"E\"]\n    assert maze_steps(big) == 398")

### Exercise: The nearest of many

A town map has `H` (hospitals), `#` (blocked) and `.` (streets). Write `distance_map(town)`: a list of lists of the distance from each cell to its nearest hospital (`None` for blocked or unreachable cells), with one multi-source BFS. Then `worst_served(town)`: the largest finite distance to a hospital, with the sorted list of cells at it (`(0, [])` if every open cell is a hospital or none is reachable).

In [ ]:
from collections import deque

def distance_map(town):
    ___

def worst_served(town):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_multi():\n    \"nearest hospitals\"\n    town = [\"H..#\",\n            \"..#.\",\n            \"...H\"]\n    assert distance_map(town) == [[0, 1, 2, None], [1, 2, None, 1], [2, 2, 1, 0]]\n    assert worst_served(town) == (2, [(0, 2), (1, 1), (2, 0), (2, 1)])\n    assert distance_map([\"#.\", \".#\"]) == [[None, None], [None, None]] and worst_served([\"H\"]) == (0, [])\n    rng = random.Random(3)\n    for _ in range(30):\n        g = [\"\".join(rng.choice(\"..#H\" if rng.random() < 0.1 else \"..#\") for _ in range(6)) for _ in range(5)]\n        dm = distance_map(g)\n        hs = [(r, c) for r in range(5) for c in range(6) if g[r][c] == \"H\"]\n        for r in range(5):\n            for c in range(6):\n                if g[r][c] == \"#\":\n                    assert dm[r][c] is None\n                elif dm[r][c] is not None:\n                    assert dm[r][c] >= min(abs(r - a) + abs(c - b) for a, b in hs)")

### Exercise: Two groups

Write `two_groups(adj)` for an undirected graph: a pair of sorted lists `(group0, group1)` such that every edge joins the two groups, or `None` if the graph isn't bipartite. In each connected piece, the first of its vertices in the dict's order goes in group 0. Then `odd_cycle_exists(adj)`.

In [ ]:
from collections import deque

def two_groups(adj):
    ___

def odd_cycle_exists(adj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_groups():\n    \"groups\"\n    square = {1: [2, 4], 2: [1, 3], 3: [2, 4], 4: [3, 1]}\n    assert two_groups(square) == ([1, 3], [2, 4]) and not odd_cycle_exists(square)\n    assert two_groups({1: [2, 3], 2: [1, 3], 3: [1, 2]}) is None and odd_cycle_exists({1: [2, 3], 2: [1, 3], 3: [1, 2]})\n    assert two_groups({\"a\": [], \"b\": [\"c\"], \"c\": [\"b\"]}) == ([\"a\", \"b\"], [\"c\"])\n    rng = random.Random(4)\n    for _ in range(100):\n        n = rng.randint(1, 8)\n        edges = {tuple(sorted(rng.sample(range(n), 2))) for _ in range(rng.randint(0, 10))} if n > 1 else set()\n        g = {v: [] for v in range(n)}\n        for a, b in edges:\n            g[a].append(b)\n            g[b].append(a)\n        res = two_groups(g)\n        if res is not None:\n            left = set(res[0])\n            assert all((a in left) != (b in left) for a, b in edges) and sorted(res[0] + res[1]) == list(range(n))\n        assert odd_cycle_exists(g) == (res is None)")

### Bug hunt: Seen too late

This BFS marks vertices as seen only when they're taken **out** of the queue, so a vertex reachable from several queued vertices is added several times. The answer is right but the work isn't: on this graph the queue receives 6 entries instead of 4, and it prints `6`. Mark vertices when they're added, so it prints `4`.

In [ ]:
from collections import deque

adj = {"A": ["B", "C", "D"], "B": ["D"], "C": ["D"], "D": []}
seen, added = set(), 0
queue = deque(["A"])
added += 1
while queue:
    v = queue.popleft()
    if v in seen:
        continue
    seen.add(v)
    for w in adj[v]:
        if w not in seen:
            queue.append(w)
            added += 1
print(added)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"mark when added\"\n    expect_output(\"4\")")

### Bug hunt: A stack, not a queue

This "BFS" takes vertices from the **end** of the list, which makes it a depth-first search, so the distances it records aren't shortest: D gets distance 3 instead of 2. Take from the front with a `deque` so it prints `{'A': 0, 'B': 1, 'C': 1, 'D': 2, 'E': 2}`.

In [ ]:
adj = {"A": ["B", "C"], "B": ["D"], "C": ["E"], "E": ["D"], "D": []}
dist = {"A": 0}
todo = ["A"]
while todo:
    v = todo.pop()
    for w in adj[v]:
        if w not in dist:
            dist[w] = dist[v] + 1
            todo.append(w)
print({k: dist[k] for k in sorted(dist, key=lambda k: (dist[k], k))})

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"first in, first out\"\n    expect_output(\"{'A': 0, 'B': 1, 'C': 1, 'D': 2, 'E': 2}\")")

### Bug hunt: Coming back to the start

This BFS never marks the **start** as seen, so in a graph with a cycle it reaches the start again and gives it a distance of 3: this prints `{'A': 3, 'B': 1, 'C': 2}`. Mark the start before the loop so it prints `{'A': 0, 'B': 1, 'C': 2}`.

In [ ]:
from collections import deque

adj = {"A": ["B"], "B": ["C"], "C": ["A"]}
dist, depth = {}, {"A": 0}
queue = deque(["A"])
while queue:
    v = queue.popleft()
    for w in adj[v]:
        if w not in dist:
            dist[w] = depth[v] + 1
            depth[w] = dist[w]
            queue.append(w)
print(dict(sorted(dist.items())))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the start is seen\"\n    expect_output(\"{'A': 0, 'B': 1, 'C': 2}\")")

### Puzzle: Breadth-first distances

Arrange the lines into a BFS that records distances. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# BFS from 1 on the undirected graph with edges 1-2, 1-3, 2-4, 3-4, 4-5, 6-7.
# Neighbours are explored in increasing order.

p = ___   # the order BFS visits the vertices
q = ___   # the distance from 1 to 5
r = ___   # the parent of 4 in the BFS tree
s = ___   # how many vertices BFS from 1 never reaches
t = ___   # True or False: the graph is bipartite

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 2, 3, 4, 5] and q == 3 and r == 2, \"p, q and r\"\n    assert s == 2 and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: doors that cost

A building's floor plan is a grid: `.` corridors, `D` doors, `#` walls, one `S` and one `E`. Walking into a corridor (or E) is free, but walking into a **door** costs 1 (you have to unlock it). Find the fewest doors needed to get from S to E.

Plain BFS counts steps, not doors, and a full weighted-path algorithm is next week. But with edge costs of only 0 and 1 there's a neat trick, **0-1 BFS**: use a deque, and when you reach a neighbour by a **free** move, add it to the **front**; by a costly move, to the **back**. The deque then always holds vertices in order of cost, at most two different costs at a time, so each vertex is finally settled in order of door count. A vertex may be added more than once (if a cheaper way is found later), so keep `best[cell]`, the fewest doors known, and skip a popped cell if it's already been settled with fewer.

Write `fewest_doors(plan)`: the fewest doors from S to E, or -1; and `doors_on_route(plan)`: the sorted list of door cells on one such best route (ties between routes may be broken any way, but the list must have exactly `fewest_doors` cells, each a door, on a real route). O(cells) time. The checks compare with a slow method on random plans.

In [ ]:
from collections import deque

def fewest_doors(plan):
    ___

def doors_on_route(plan):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, heapq\ndef slow(plan):\n    rows, cols = len(plan), len(plan[0])\n    s = next((r, c) for r in range(rows) for c in range(cols) if plan[r][c] == \"S\")\n    best, heap = {s: 0}, [(0, s)]\n    while heap:\n        d, (r, c) = heapq.heappop(heap)\n        if plan[r][c] == \"E\":\n            return d\n        if d > best[(r, c)]:\n            continue\n        for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):\n            if 0 <= nr < rows and 0 <= nc < cols and plan[nr][nc] != \"#\":\n                nd = d + (plan[nr][nc] == \"D\")\n                if nd < best.get((nr, nc), 10**9):\n                    best[(nr, nc)] = nd\n                    heapq.heappush(heap, (nd, (nr, nc)))\n    return -1\ndef test_doors():\n    \"doors\"\n    plan = [\"S.D..\",\n            \"##D#.\",\n            \"E.D.D\",\n            \".....\"]\n    assert fewest_doors(plan) == 2\n    assert fewest_doors([\"SDE\"]) == 1 and fewest_doors([\"S#E\"]) == -1 and fewest_doors([\"SE\"]) == 0 and doors_on_route([\"SE\"]) == []\n    route = doors_on_route(plan)\n    assert len(route) == 2 and all(plan[r][c] == \"D\" for r, c in route)\ndef test_random():\n    \"against a slow method\"\n    rng = random.Random(5)\n    for _ in range(200):\n        R, C = rng.randint(1, 6), rng.randint(2, 6)\n        cells = [[rng.choice(\"..D#\") for _ in range(C)] for _ in range(R)]\n        cells[0][0], cells[R - 1][C - 1] = \"S\", \"E\"\n        plan = [\"\".join(row) for row in cells]\n        want = slow(plan)\n        assert fewest_doors(plan) == want, plan\n        route = doors_on_route(plan)\n        if want == -1:\n            assert route is None\n        else:\n            assert len(route) == want and all(plan[r][c] == \"D\" for r, c in route)")

## Warm-up

D is reachable two ways (through B and through C), but it's added to the queue once: the first time it's seen. And its distance, 2, is the length of the shortest path from A.

## The algorithm

Breadth-first search from a start vertex s:

1. Mark s as seen and put it in a queue.
2. While the queue isn't empty: take the vertex at the front; for each neighbour not yet seen, mark it seen and add it to the back.

Two details matter:

- **Mark vertices when you add them to the queue, not when you take them out.** Otherwise a vertex reachable from several queued vertices is added several times; it's still correct, but the queue can grow far beyond V and the work far beyond O(V + E).
- **Use a `deque`** (`popleft` is O(1)); `list.pop(0)` is O(n), which makes BFS quadratic.

Each vertex enters the queue once and each edge is looked at once from each end (twice for undirected graphs), so BFS is **O(V + E)** time and O(V) extra space.

BFS finds only what's **reachable** from s. Vertices in other pieces of the graph are never seen, which is itself useful: Thursday uses it to find a graph's connected pieces.

The method is usually credited to Edward F. Moore, who described it in 1959 for finding the shortest path through a maze; C. Y. Lee published it independently in 1961 for routing wires on circuit boards.

## Shortest paths

In an **unweighted** graph (every edge counts as 1), the order in which BFS first reaches a vertex is the order of distance: everything at distance 1 is reached before anything at distance 2, and so on. So setting `dist[w] = dist[v] + 1` when w is first seen gives the shortest distance from the start to every reachable vertex.

To get the **path** itself, record each vertex's **parent**: the vertex it was first reached from. Then walk back from the target to the start and reverse. The parent links form a tree, the **BFS tree**, holding one shortest path to every reachable vertex.

Stopping as soon as the goal is taken from the queue saves work when you only need one target. BFS is right **only** when every edge counts the same; with different weights (minutes, kilometres), the path with fewest edges may not be the quickest. Next week's Dijkstra's algorithm handles weights.

## Grids and puzzles

A grid is an implicit graph (Monday), so BFS works on it unchanged, with cells as vertices: it finds the shortest route through a maze in O(rows × columns). The `dist` dict doubles as the seen set.

Puzzle **states** are graphs too. Take the classic jug puzzle: a 3-litre jug and a 5-litre jug, a tap, a drain, and the task of measuring exactly 4 litres. A state is `(litres in small, litres in large)`; a move is filling a jug, emptying one, or pouring one into the other until one is empty or full. BFS from `(0, 0)` finds the fewest moves to any state with 4 litres in it.

## Many starting points

"How far is every house from its **nearest** fire station?" Running BFS from each station and taking the minimum would cost O(stations × (V + E)). Instead, start one BFS with **all** the stations in the queue at distance 0. The rings spread out from all of them together, and each house is first reached from whichever station is nearest. That's **multi-source BFS**: still O(V + E).

## Two colours

Can a graph's vertices be split into two groups so that every edge goes **between** the groups, never within one? Such a graph is **bipartite**: people and the projects they work on, students and the courses they take. BFS answers it: colour the start 0, its neighbours 1, their neighbours 0, and so on; if an edge ever joins two vertices of the same colour, it's impossible. A graph is bipartite exactly when it has no cycle of odd length (a triangle can't be two-coloured).

## Practice

## Quiz

## Daily challenge

## Before you go

On a map of your area, pick a place and colour everything one road junction away, then two, then three: that's BFS by hand. Tomorrow: depth-first search, which goes as deep as it can before backing up, and what it reveals about a graph's structure.